In [3]:
import sys
sys.path.append('utilities/')
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import torch
from sentence_transformers import SentenceTransformer
from joblib import dump
from openai import OpenAI
from tqdm import tqdm
from mmd import MMD
import re
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import ot

/Users/pranitgunjal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [168]:
orig_test_path = 'data/initial_datasets/dota2/dota2_grouped_train.csv'
orig_train_path = 'data/initial_datasets/dota2/dota2_grouped_test.csv'
gen_path = 'data/generated/dota2/conversational_generations/grouped_conversational_var_len.csv'
n = 250

## Temp

In [88]:
train_df = pd.read_csv(orig_train_path)
test_df = pd.read_csv(orig_test_path)
gen_df = pd.read_csv(gen_path)

In [28]:
train_df['labels'] = train_df['labels'].replace({-1: 1, 1: 0})

In [27]:
test_df['labels'] = test_df['labels'].replace({-1: 1, 1: 0})

In [79]:
gen_df['labels'] = gen_df['labels'].replace({-1: 1, 1: 0})

In [89]:
test_df

,messages,labels,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 645,Unnamed: 646,Unnamed: 647,Unnamed: 648,Unnamed: 649,Unnamed: 650,Unnamed: 651,Unnamed: 652,Unnamed: 653,Unnamed: 654
0,On this holiest of Jewish religious observance...,Republican,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,THANK YOU. YOU HAVE NO IDEA HOW GREAT IT IS TO...,Democrat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"IT LOOKS LIKE IT WILL BE A LONG NIGHT, BUT I F...",Democrat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,So we're going to Wisconsin. We have a big cro...,Republican,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Well, thank you, Jim Bridenstine. Thank you fo...",Republican,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,Thank you all for coming out on this blustery ...,Republican,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
280,"Well, hello, Kentucky! (Applause.) Thank you f...",Republican,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
281,HELLO FOLKS HOW ARE YOU? THANK YOU. THANK YOU....,Democrat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
282,"Thank you, Secretary Chao. Thank you for those...",Republican,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
train_df

,messages,labels
0,First is the worst,1
1,Our education system has been a complete and u...,1
2,"The fuck you call me!? A cunt!? Damn man, didn...",1
3,It will probably take him some time to figure ...,0
4,Somebody is really insecure about their career...,1
...,...,...
995,Be glad you don't know the answer.,0
996,I’m laughing more that I feel I should st this...,0
997,I wouldnt necessarily call you and addict but ...,1
998,I'm genuinely interested in the responses to t...,0


In [86]:
gen_df

,messages,labels
0,I’m so glad I found this community! Everyone h...,0
1,"Ugh, this really sucks. I can't believe people...",1
2,"Wow, I just finished that book and it was abso...",0
3,"I thought my day couldn't get any worse, then ...",1
4,I can't express how happy I am. Everything wen...,0
...,...,...
1105,"Another day, another boring meeting 😒",1
1106,"So glad I went for a walk today, feeling refre...",0
1107,That game last night was pretty disappointing 😞,1
1108,Just finished a great book and loved every page!,0


In [ ]:
train_df.to_csv(orig_train_path, index=False).sample(n=n, random_state=42)
test_df.to_csv(orig_test_path, index=False)
gen_df.to_csv(gen_path, index=False)

## End Temp

In [169]:
orig_df = pd.read_csv(orig_test_path).sample(n=n, random_state=42)
orig_sample_df = pd.read_csv(orig_train_path).sample(n=n, random_state=42)
gen_df = pd.read_csv(gen_path).sample(n=n, random_state=42)

In [164]:
orig_df

,match,player,messages,labels
142,99,2,GG,0
6,62,6,"""come on"". ""rampage incoming""",0
97,73,6,"""stay afk""",0
60,37,4,"HAHAHAHA. ""HAHAHAHAHA"". HAHAHAH",0
112,35,5,"""dusa, es and sf won you the game"". ""dog shit""...",0
...,...,...,...,...
106,10,3,"""Why"". ""Fuck"". Good. wahahaha. ""loser"". ""Don't...",1
14,27,6,gg,0
92,33,3,"""easy snake faggot"". ""10 lvl snake faggot dies...",1
179,10,6,"""my tb always dying"". L O L. ""stupid Dota"". TN...",0


In [165]:
gen_df

,messages,labels
521,You ARE SUCH a fucking garbage player. REPORT ...,1
737,How do you manage to screw up every game I’m w...,1
740,"GG everyone, nice match.Jungle is stacked if y...",0
660,What are you doing?This game is over if you ke...,1
411,guys back off!they’re closing in.regroup high ...,0
...,...,...
109,Can you stop feeding? It's like you learned no...,1
430,That Eul's was on point!Let's focus Luna next....,0
77,"Why the hell go jungle, you're useless. How ca...",1
84,nice try team.we almost had that one.great pla...,0


In [166]:
orig_sample_df

,match,player,messages,labels
142,71,8,"DOTA ')'). ""kaivgpshshchfknrzeghsh"". XD. ""fuck...",0
6,16,9,"so close... indeed. just end. ""blademail and u...",0
97,18,9,"wp!. ""well played!"". ""what the damn fuck"". HAH...",0
60,64,5,"""eat bitches"". gg",0
112,74,0,"""HAHAHAHAHA"". ""PLEASE LEAVE"". ""slark report"". GG",0
...,...,...,...,...
106,59,2,"""suck?"". ""Hahahahahahahahahahahahahahahahahaha...",0
14,58,1,thank you,0
92,81,3,"""fuck you"". Thank you. ""eat eggs"". gg. haha. ""...",1
179,81,1,"""Ahmad Maslan continues"". ""that's it"". ""you ta...",1


In [170]:
sentence_transformer = SentenceTransformer('all-mpnet-base-v2')

X = np.array(sentence_transformer.encode(orig_df['messages'].to_list()))
Y = np.array(sentence_transformer.encode(gen_df['messages'].to_list()))

def report_MMD(X, Y, normalize=False):
    
    tensorX = torch.tensor(X)
    tensorY = torch.tensor(Y)
    rbf_mmd = MMD(tensorX, tensorY, "rbf")
    scale_mmd = MMD(tensorX, tensorY, "multiscale")


    if normalize:
        return (rbf_mmd.item() / np.sqrt((1.0 / X.shape[0]) + (1.0 / Y.shape[0]))), (scale_mmd.item() / np.sqrt(1.0 / X.shape[0] + 1.0 / Y.shape[0]))
    else:
        return rbf_mmd.item(), scale_mmd.item()
    
print("Reported MMD: ")
print(report_MMD(X, Y, normalize=True))
print()

def wasserstein(X, Y):
    
    a = np.ones((X.shape[0],)) / X.shape[0]
    b = np.ones((Y.shape[0],)) / Y.shape[0]

    M = ot.dist(X, Y)
    M /= M.max()

    return ot.emd2(a, b, M)

print("Reported Wasserstein")
print(wasserstein(X, Y))
print()
train_df = gen_df
test_df = orig_df
train_df['labels'] = train_df['labels'].astype(int)
test_df['labels'] = test_df['labels'].astype(int)

X_train = train_df['messages']
X_test = test_df['messages']

y_train = train_df['labels']
y_test = test_df['labels']

X_train = np.array(sentence_transformer.encode(X_train.to_list()))
X_test = np.array(sentence_transformer.encode(X_test.to_list()))

model = svm.SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

print('Trained on just Synthetic')

y_pred = model.predict(X_train)
train_acc = accuracy_score(y_pred, y_train)
print(f'Train acc: {train_acc}')

y_pred = model.predict(X_test)
test_acc = accuracy_score(y_pred, y_test)
print(f'Test acc: {test_acc}')
print()
print(f"Precision Score: {precision_score(y_test.to_list(), y_pred)}")
print(f"Recall Score: {recall_score(y_test.to_list(), y_pred)}")

y_prob = model.predict_proba(X_test)[:, 1]
print(f"Roc_auc_score: {roc_auc_score(y_test, y_prob)}")
print()


train_df = pd.concat([gen_df.sample(frac=0.9, random_state=42), orig_sample_df.sample(frac=0.1, random_state=42)]).sample(frac=1, random_state=42).reset_index(drop=True)
test_df = orig_df.sample(frac=1, random_state=42).reset_index(drop=True)

X_train = train_df['messages']
X_test = test_df['messages']

y_train = train_df['labels']
y_test = test_df['labels']

X_train = np.array(sentence_transformer.encode(X_train.to_list()))
X_test = np.array(sentence_transformer.encode(X_test.to_list()))

model = svm.SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

print("Trained on just Synthetic + 10% Real")

y_pred = model.predict(X_train)
train_acc = accuracy_score(y_pred, y_train)
print(f'Train acc: {train_acc}')

y_pred = model.predict(X_test)
test_acc = accuracy_score(y_pred, y_test)
print(f'Test acc: {test_acc}')
print()
print(f"Precision Score: {precision_score(y_test.to_list(), y_pred)}")
print(f"Recall Score: {recall_score(y_test.to_list(), y_pred)}")

y_prob = model.predict_proba(X_test)[:, 1]
print(f"Roc_auc_score: {roc_auc_score(y_test, y_prob)}")


Reported MMD: 
(np.float64(0.6395309319040173), np.float64(4.378518457304397))

Reported Wasserstein
0.5366568835973742

Trained on just Synthetic
Train acc: 0.948
Test acc: 0.676

Precision Score: 0.16455696202531644
Recall Score: 0.4642857142857143
Roc_auc_score: 0.6327220077220078

Trained on just Synthetic + 10% Real
Train acc: 0.928
Test acc: 0.824

Precision Score: 0.23333333333333334
Recall Score: 0.25
Roc_auc_score: 0.6554054054054055
